# Phase 7 · P7-0 · look at the data + BM25 baseline (CPU, no language model, ~30 min)
**Accelerator: None.** Needs only the Kaggle secret `capstone` (or `HF_TOKEN`). This is a **baseline, not one of the
required experiments**: it uses no Phase-1 model. It answers the questions the plan depends on: what the fields look
like, whether citation markers survive in the test queries, whether pools overlap, whether dates can rule candidates
out, and how far word n-gram BM25 goes. Settings are chosen on the **training queries**, then reported once on dev.

In [ ]:
# 1 · Settings
import os, sys, json, time
import numpy as np
OUT = "/kaggle/working"; os.makedirs(OUT + "/p7src", exist_ok=True); os.chdir(OUT); sys.path.insert(0, OUT + "/p7src")

In [ ]:
# 2 · Hugging Face token (Kaggle secret "capstone" or "HF_TOKEN")
from huggingface_hub import login, whoami
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    for _n in ("capstone", "HF_TOKEN"):
        try:
            HF_TOKEN = UserSecretsClient().get_secret(_n)
            if HF_TOKEN: break
        except Exception:
            pass
except ImportError:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
assert HF_TOKEN, "add the Hugging Face token as a Kaggle secret and attach it"
login(token=HF_TOKEN); print("Hugging Face user:", whoami(token=HF_TOKEN)["name"])

In [ ]:
%%writefile /kaggle/working/p7src/pcr_lex.py
# Phase 7 (Prior Case Retrieval): data reading, data probes, BM25 baseline. No language model is used here.
#   Micro-F1@7 = 2 * hits / (7 * queries + gold links): with exactly 7 answers per query it only depends on the
#   number of gold cases inside the top 7, so the order inside the top 7 does not change the score.
import json
import re
from collections import Counter

import numpy as np
import scipy.sparse as sp

K = 7
ID_KEYS = ("id", "query_id", "case_id", "doc_id", "candidate_id", "qid", "cid")
TEXT_KEYS = ("text", "sentences", "document", "doc", "content")
GOLD_KEYS = ("relevant_candidates", "relevant", "relevant_ids", "gold", "labels", "citations")
MARK = re.compile(r"<\s*/?\s*citation\s*/?\s*>|\[\s*citation\s*\]", re.I)
WORD = re.compile(r"[a-z0-9]+")
YEAR = re.compile(r"\b(1[89]\d\d|20[0-2]\d)\b")


def as_text(t):
    return [str(x) for x in t] if isinstance(t, list) else [str(t)]


def parse(raw):
    """list or dict of records (or id -> text) -> {id: dict(sents=[...], gold=[ids] or None)}"""
    if isinstance(raw, dict) and len(raw) <= 3:
        for k in ("data", "queries", "candidates", "cases", "documents"):
            if isinstance(raw.get(k), (list, dict)): raw = raw[k]; break
    items = raw.items() if isinstance(raw, dict) else ((None, r) for r in raw)
    out = {}
    for key, r in items:
        if not isinstance(r, dict): r = dict(text=r)
        did = key if key is not None else next(r[k] for k in ID_KEYS if k in r)
        text = next((r[k] for k in TEXT_KEYS if r.get(k)), "")
        gold = next((r[k] for k in GOLD_KEYS if k in r), None)
        out[str(did)] = dict(sents=as_text(text), gold=None if gold is None else [str(g) for g in gold])
    return out


def words(sents):
    return WORD.findall(MARK.sub(" ", " ".join(sents)).lower())


def context(sents, w=1):
    """only the sentences holding a citation marker, +- w sentences (where the query uses its prior cases)"""
    hit = [i for i, s in enumerate(sents) if MARK.search(s)]
    keep = sorted({j for i in hit for j in range(max(0, i - w), min(len(sents), i + w + 1))})
    return [sents[j] for j in keep]


def grams(w, n):
    return w if n == 1 else [" ".join(w[i:i + n]) for i in range(len(w) - n + 1)]


def bm25(query_words, doc_words, n, k1=1.2, b=0.75):
    """scores (queries, docs). Only n-grams that occur in some query are indexed, which keeps long n-grams in memory.
    Every distinct query n-gram counts once (long queries are not dominated by their repeated phrases)."""
    vocab = {}
    qr, qc = [], []
    for i, w in enumerate(query_words):
        for g in set(grams(w, n)):
            qr.append(i); qc.append(vocab.setdefault(g, len(vocab)))
    Q = sp.csr_matrix((np.ones(len(qr), dtype=np.float32), (qr, qc)), shape=(len(query_words), len(vocab)))
    dr, dc, dv, dl = [], [], [], []
    for i, w in enumerate(doc_words):
        c = Counter(grams(w, n)); dl.append(sum(c.values()))
        for g, f in c.items():
            j = vocab.get(g)
            if j is not None: dr.append(i); dc.append(j); dv.append(f)
    D = sp.csr_matrix((np.array(dv, dtype=np.float32), (dr, dc)), shape=(len(doc_words), len(vocab)))
    dl = np.maximum(1.0, np.array(dl, dtype=np.float32)); N = len(doc_words)
    df = np.asarray((D > 0).sum(0)).ravel()
    idf = np.log(1 + (N - df + 0.5) / (df + 0.5))
    norm = k1 * (1 - b + b * dl / dl.mean())
    W = D.tocoo()
    W.data = W.data * (k1 + 1) / (W.data + norm[W.row]) * idf[W.col]
    return np.asarray((Q @ W.tocsr().T).todense(), dtype=np.float32)


def zscore(S):
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-6)


def rank(S, cids, qids, drop_self=True):
    """scores -> {query: top K candidate ids}; a query is never its own answer"""
    out = {}
    for i, q in enumerate(qids):
        o = np.argsort(-S[i], kind="stable")
        out[q] = [cids[j] for j in o if not (drop_self and cids[j] == q)][:K]
    return out


def micro(pred, gold):
    """exactly the leaderboard definition: gold is neither truncated nor padded"""
    hit = sum(len(set(pred[q][:K]) & set(gold[q])) for q in pred)
    p = hit / max(1, sum(len(pred[q][:K]) for q in pred)); r = hit / max(1, sum(len(gold[q]) for q in pred))
    return dict(f1=2 * p * r / max(1e-9, p + r), p=p, r=r, hits=hit)


def per_query_hits(pred, gold):
    return np.array([len(set(pred[q][:K]) & set(gold[q])) for q in pred], dtype=float)


def paired(a, b, gold, n=2000, seed=0):
    """a minus b in micro-F1 with a query-level bootstrap -> (difference, 2.5%, 97.5%)"""
    qs = list(a); ha, hb = per_query_hits(a, gold), per_query_hits(b, {q: gold[q] for q in b})
    ng = np.array([len(gold[q]) for q in qs], dtype=float); rng = np.random.default_rng(seed)
    f = lambda h, g: 2 * h.sum() / (K * len(h) + g.sum())
    d = [f(ha[i], ng[i]) - f(hb[i], ng[i]) for i in (rng.integers(0, len(qs), len(qs)) for _ in range(n))]
    return f(ha, ng) - f(hb, ng), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))


def years(sents):
    y = [int(m) for m in YEAR.findall(" ".join(sents))]
    return max(y) if y else None


def check_submission(pred, Q, C):
    assert set(pred) == set(Q), "a test query is missing"
    for q, c in pred.items():
        assert len(c) == K and len(set(c)) == K and all(x in C for x in c), q


def write_submission(pred, path_zip):
    import os, zipfile
    d = os.path.dirname(path_zip) or "."; pj = os.path.join(d, "prediction.json")
    json.dump(pred, open(pj, "w"));
    with zipfile.ZipFile(path_zip, "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, arcname="prediction.json")
    return path_zip

In [ ]:
# 3 · Load the three splits, print the raw record layout
import pcr_lex as lx
from huggingface_hub import hf_hub_download
REPO = "Exploration-Lab/CS-781-Capstone"
get = lambda s, f: json.load(open(hf_hub_download(REPO, f"Phase_7_PCR/{s}/{f}", repo_type="dataset", token=HF_TOKEN), encoding="utf-8"))
def show(name, raw):
    r = raw[0] if isinstance(raw, list) else next(iter(raw.items()))
    print(f"{name}: {type(raw).__name__} of {len(raw):,} | first record: {str(r)[:350]}")
Q, C = {}, {}
for s, a in (("Train", "train"), ("Dev", "dev"), ("Test", "test")):
    rq, rc = get(s, f"{a}.json"), get(s, f"{a}_candidates.json")
    show(a + " queries", rq); show(a + " candidates", rc)
    Q[a], C[a] = lx.parse(rq), lx.parse(rc); del rq, rc
gold = {a: {q: r["gold"] for q, r in Q[a].items() if r["gold"] is not None} for a in Q}
assert len(gold["train"]) == len(Q["train"]) and len(gold["dev"]) == len(Q["dev"]), "gold field not found: see the first record above"
assert not gold["test"], "test queries carry labels?"

In [ ]:
# 4 · Data probes
tot = lambda r: sum(len(s.split()) for s in r["sents"])
for a in Q:
    wq, wc = np.array([tot(r) for r in Q[a].values()]), np.array([tot(r) for r in C[a].values()])
    mark = np.array([sum(bool(lx.MARK.search(s)) for s in r["sents"]) for r in Q[a].values()])
    line = (f"{a:<5}: {len(Q[a]):,} queries, words median {int(np.median(wq)):,} max {wq.max():,} | {len(C[a]):,} candidates, words median {int(np.median(wc)):,} max {wc.max():,} | "
            f"sentences with a citation marker per query: median {int(np.median(mark))}, queries without any {(mark == 0).sum()}")
    if gold[a]:
        n = np.array([len(g) for g in gold[a].values()]); inpool = np.mean([x in C[a] for g in gold[a].values() for x in g])
        line += f" | gold per query mean {n.mean():.2f}, median {int(np.median(n))}, min {n.min()}, max {n.max()}, share <= 7 {np.mean(n <= 7):.2f} | gold in pool {inpool:.3f}"
        print(line, f"| best possible micro-F1@7 {2 * np.minimum(n, 7).sum() / (7 * len(n) + n.sum()):.3f}, random {7 / len(C[a]) * n.mean() * 2 * len(n) / (7 * len(n) + n.sum()):.4f}")
    else: print(line)
print("\npool overlap:", {f"{a}&{b}": len(set(C[a]) & set(C[b])) for a, b in (("train", "dev"), ("train", "test"), ("dev", "test"))},
      "| queries inside their own pool:", {a: len(set(Q[a]) & set(C[a])) for a in Q},
      "| query overlap:", {f"{a}&{b}": len(set(Q[a]) & set(Q[b])) for a, b in (("train", "dev"), ("train", "test"), ("dev", "test"))})
from collections import Counter
seen = Counter(x for g in gold["train"].values() for x in g)
print("train gold: distinct cited cases", len(seen), "| top 10 cases hold", f"{sum(c for _, c in seen.most_common(10)) / sum(seen.values()):.3f}", "of links",
      "| dev gold links already cited in train:", f"{np.mean([x in seen for g in gold['dev'].values() for x in g]):.3f}")
# can dates rule candidates out? (year = latest year named in the text; a prior case cannot be newer than its citing case)
yq = {a: {q: lx.years(r["sents"]) for q, r in Q[a].items()} for a in Q}; yc = {a: {c: lx.years(r["sents"]) for c, r in C[a].items()} for a in Q}
for a in ("train", "dev"):
    pairs = [(yq[a][q], yc[a][x]) for q, g in gold[a].items() for x in g if x in yc[a] and yq[a][q] and yc[a][x]]
    print(f"{a}: gold candidate newer than its query in {np.mean([c > q for q, c in pairs]):.4f} of links | an average query would remove "
          f"{np.mean([np.mean([c is not None and yq[a][q] is not None and c > yq[a][q] for c in yc[a].values()]) for q in list(gold[a])[:100]]):.3f} of the pool")
print("\nexample sentences with a marker (test):")
for r in list(Q["test"].values())[:2]:
    print("  ", [s[:160] for s in r["sents"] if lx.MARK.search(s)][:3])

In [ ]:
# 5 · BM25 grid. Settings are picked on the TRAINING queries; dev is only reported.
words = {a: dict(q=[lx.words(r["sents"]) for r in Q[a].values()], c=[lx.words(r["sents"]) for r in C[a].values()],
                 ctx=[lx.words(lx.context(r["sents"]) or r["sents"]) for r in Q[a].values()]) for a in Q}
qids = {a: list(Q[a]) for a in Q}; cids = {a: list(C[a]) for a in C}
def f(a, S): return lx.micro(lx.rank(S, cids[a], qids[a]), gold[a])
scores = {a: {} for a in Q}; t0 = time.time()
for view in ("q", "ctx"):
    for n in (1, 2, 3, 4, 5):
        for a in ("train", "dev"):
            scores[a][view, n] = lx.bm25(words[a][view], words[a]["c"], n)
        tr, de = f("train", scores["train"][view, n]), f("dev", scores["dev"][view, n])
        print(f"{'whole query' if view == 'q' else 'citation context':<17} {n}-gram | train F1 {tr['f1']:.4f} | dev F1 {de['f1']:.4f} (P {de['p']:.4f} R {de['r']:.4f}) | {(time.time() - t0) / 60:.1f} min")

In [ ]:
# 6 · Sum of z-scored views: best set on train (greedy), then one dev report
keys = sorted(scores["train"], key=lambda k: -f("train", scores["train"][k])["f1"]); chosen = keys[:1]
fuse = lambda a, ks: sum(lx.zscore(scores[a][k]) for k in ks)
best = f("train", fuse("train", chosen))["f1"]
while True:
    tryk = [(f("train", fuse("train", chosen + [k]))["f1"], k) for k in keys if k not in chosen]
    if not tryk or max(tryk)[0] <= best + 1e-4: break
    best, k = max(tryk); chosen.append(k); print(f"+ {k}: train F1 {best:.4f}")
single = lx.rank(scores["dev"][keys[0]], cids["dev"], qids["dev"]); mix = lx.rank(fuse("dev", chosen), cids["dev"], qids["dev"])
print(f"\nbest single on train: {keys[0]} | dev {lx.micro(single, gold['dev'])}")
print(f"fusion {chosen} | dev {lx.micro(mix, gold['dev'])}")
print("fusion minus single on dev (difference, 95%% interval): %.4f [%.4f, %.4f]" % lx.paired(mix, single, gold["dev"]))

In [ ]:
# 7 · Test zip (a calibration of dev against the leaderboard - no Phase-1 model, so NOT one of the required experiments)
S = sum(lx.zscore(lx.bm25(words["test"][v], words["test"]["c"], n)) for v, n in chosen)
pred = lx.rank(S, cids["test"], qids["test"]); lx.check_submission(pred, Q["test"], C["test"])
print("SUBMIT:", lx.write_submission(pred, f"{OUT}/submission_p7_0_bm25_baseline.zip"), "|", len(pred), "queries | ", list(pred.items())[0])